# Motivating Example: Why Gradient-Based Optimization?

**Scene:** Duke — single transmitter, obstructed square coverage zone  
**Frequency:** 3.5 GHz

Benchmarks two conventional methods to establish that better approaches are needed:

| Baseline | Category | Method |
|---|---|---|
| `uma_naive` | Empirical | 3GPP TR 38.901 geometric look-at + fixed electrical downtilt |
| `random_search` | Brute-force | Exhaustive random candidate sampling over boresight and position |

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
SCENE_XML_PATH = "../scene/scenes/Duke/scene.xml"
CARRIER_HZ     = 3.5e9

# Transmitter
TX_NAME        = "gnb"
TX_BUILDING_ID = 33          # Verify with zone visualization below
TX_HEIGHT_M    = 10.0

# ── ZONE CONFIGURATION ──────────────────────────────────────────────────────
# Coordinates are scene-local meters. Run the visualization cell to verify.
ZONE_CENTER   = [-200.0, 300.0]   # [x, y]
ZONE_WIDTH_M  = 200.0
ZONE_HEIGHT_M = 200.0

# Radio map grid
MAP_CONFIG = {
    'center':        [0.0, 0.0, 0.0],
    'size':          [1400, 1400],
    'cell_size':     (0.5, 0.5),
    'ground_height': 0.0,
}

# Experiment
NOISE_POWER   = 1e-10
JITTER_SEED   = 42         # Fixed seed → reproducible initial perturbation
JITTER_MAG    = 1e-4       # Degrees — also used as convergence tolerance
BASELINES     = ["uma_naive", "random_search"]
OUTPUT_PATH   = "../scripts/report/motivating_example_results.json"

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
import mitsuba as mi

try:
    import sionna.rt
except ImportError:
    os.system("pip install sionna-rt")
    import sionna.rt

from sionna.rt import load_scene, AntennaArray
from sionna.rt.antenna_pattern import antenna_pattern_registry

from scene_parser import extract_building_info
from tx_placement import TxPlacement
from boresight_pathsolver import create_zone_mask
from angle_utils import compute_initial_angles_from_position, azimuth_elevation_to_yaw_pitch
from multi_tx_optimizer import TxConfig
from experiment_runner import (
    ExperimentConfig, run_experiment_suite,
    compare_all_results, plot_cdf, plot_metric_bars,
)

scene = load_scene(SCENE_XML_PATH)
scene.frequency = CARRIER_HZ

single_el = np.array([[0.0, 0.0, 0.0]])
scene.tx_array = AntennaArray(
    antenna_pattern=antenna_pattern_registry.get("tr38901")(polarization="V"),
    normalized_positions=single_el.T,
)
scene.rx_array = AntennaArray(
    antenna_pattern=antenna_pattern_registry.get("iso")(polarization="V"),
    normalized_positions=single_el.T,
)
for rm in scene.radio_materials.values():
    rm.scattering_coefficient = 0.4

building_info = extract_building_info(SCENE_XML_PATH, verbose=False)
print(f"Scene loaded  |  {CARRIER_HZ/1e9:.1f} GHz  |  {len(building_info)} buildings")

In [ ]:
# ── ZONE MASK ────────────────────────────────────────────────────────────────
zone_params = {'center': ZONE_CENTER, 'width': ZONE_WIDTH_M, 'height': ZONE_HEIGHT_M}

zone_mask, look_at_pos, zone_stats = create_zone_mask(
    map_config=MAP_CONFIG,
    zone_type='box',
    zone_params=zone_params,
    target_height=1.5,
    scene_xml_path=SCENE_XML_PATH,
    exclude_buildings=True,
)
zone_masks = {TX_NAME: zone_mask}

print(f"Zone cells:      {zone_stats['num_cells']}")
print(f"Zone centroid:   {zone_stats['centroid_xy']}")
print(f"Look-at (naive): {zone_stats['look_at_xyz']}")

In [ ]:
# ── TX PLACEMENT ─────────────────────────────────────────────────────────────
# Places TX on the building edge closest to the zone centroid.
tx_placer = TxPlacement(scene, TX_NAME, SCENE_XML_PATH, TX_BUILDING_ID, offset=TX_HEIGHT_M)
tx_placer.set_rooftop_zone_facing(zone_stats["centroid_xy"])

tx = scene.get(TX_NAME)
tx_pos = tx.position.numpy().flatten().tolist()
print(f"TX on building {TX_BUILDING_ID} at ({tx_pos[0]:.2f}, {tx_pos[1]:.2f}, {tx_pos[2]:.2f})")

In [ ]:
# ── INITIAL JITTER ───────────────────────────────────────────────────────────
# Small fixed-seed perturbation around the geometric look-at direction.
# Convergence is defined as per-iteration angle change < JITTER_MAG degrees.
base_az, base_el = compute_initial_angles_from_position(tx_pos, zone_stats["look_at_xyz"])

rng = np.random.default_rng(JITTER_SEED)
initial_az = base_az + float(rng.uniform(-JITTER_MAG, JITTER_MAG))
initial_el = base_el + float(rng.uniform(-JITTER_MAG, JITTER_MAG))

yaw_r, pitch_r = azimuth_elevation_to_yaw_pitch(initial_az, initial_el)
tx.orientation = mi.Point3f(yaw_r, pitch_r, 0.0)

print(f"Geometric look-at:  Az = {base_az:.4f}°,  El = {base_el:.4f}°")
print(f"After jitter:       Az = {initial_az:.6f}°, El = {initial_el:.6f}°")

In [ ]:
# ── ZONE VISUALIZATION ───────────────────────────────────────────────────────
_cx, _cy, _ = MAP_CONFIG['center']
_w, _h = MAP_CONFIG['size']
extent = [_cx - _w/2, _cx + _w/2, _cy - _h/2, _cy + _h/2]

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(np.ma.masked_where(zone_mask == 0, zone_mask),
          origin='lower', extent=extent, cmap='Blues', vmin=0, vmax=1, alpha=0.5)
for bdata in building_info.values():
    verts = bdata['vertices'][:, :2]
    ax.add_patch(MplPolygon(verts, closed=True, facecolor='gray',
                            edgecolor='black', linewidth=0.8, alpha=0.4))
ax.plot(tx_pos[0], tx_pos[1], '^', color='red', markersize=14,
        markeredgecolor='black', label=f'TX: {TX_NAME} (bldg {TX_BUILDING_ID})', zorder=5)
ax.plot(*zone_stats['centroid_xy'], 'o', color='steelblue', markersize=10,
        markeredgecolor='black', label='Zone centroid', zorder=5)
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Coverage Zone — Motivating Example (Duke, 3.5 GHz)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── CONVERGENCE HELPER ───────────────────────────────────────────────────────
# Gradient methods:  first iteration i where both |az[i]-az[i-1]| and
#                    |el[i]-el[i-1]| are below `tol`.
# Gradient-free:     total angular displacement from initial to final solution.

def check_convergence(suite_output, baseline_id, tx_name, tol=1e-4):
    entry = suite_output["results"].get(baseline_id, {})
    opt   = entry.get("optimizer_result")
    if not isinstance(opt, dict):
        return {"status": "error"}
    tx_res  = opt.get(tx_name, {})
    az_hist = tx_res.get("az_history", [])
    el_hist = tx_res.get("el_history", [])
    if len(az_hist) > 1:
        for i in range(1, len(az_hist)):
            if abs(az_hist[i] - az_hist[i-1]) < tol and abs(el_hist[i] - el_hist[i-1]) < tol:
                return {"status": "converged", "iteration": i}
        return {"status": "no convergence", "iterations": len(az_hist)}
    init_angles = suite_output["metadata"]["initial_angles"].get(tx_name, [None, None])
    best_angles = tx_res.get("best_angles", [None, None])
    if None in list(init_angles) + list(best_angles):
        return {"status": "error"}
    d_az = abs(best_angles[0] - init_angles[0])
    d_el = abs(best_angles[1] - init_angles[1])
    label = "no movement" if d_az < tol and d_el < tol else "displaced"
    return {"status": label, "Δaz_deg": round(d_az, 4), "Δel_deg": round(d_el, 4)}

In [ ]:
# ── RUN SUITE ────────────────────────────────────────────────────────────────
tx_config = TxConfig(
    name=TX_NAME,
    building_id=TX_BUILDING_ID,
    zone_params=zone_params,
    tx_height_offset=TX_HEIGHT_M,
    num_sample_points=256,
)
exp_config = ExperimentConfig(
    baselines=BASELINES,
    noise_power=NOISE_POWER,
    output_path=OUTPUT_PATH,
)
suite_output = run_experiment_suite(
    scene=scene,
    tx_configs=[tx_config],
    map_config=MAP_CONFIG,
    scene_xml_path=SCENE_XML_PATH,
    zone_masks=zone_masks,
    exp_config=exp_config,
)

In [ ]:
# ── CONVERGENCE + RESULTS TABLE ──────────────────────────────────────────────
print(f"\n{'─'*55}")
print(f"  Convergence Analysis  (tol = {JITTER_MAG:.0e} deg)")
print(f"{'─'*55}")
for bid in BASELINES:
    conv = check_convergence(suite_output, bid, TX_NAME, tol=JITTER_MAG)
    print(f"  {bid:<20s}  {conv}")

df = compare_all_results(suite_output)

In [ ]:
# ── VISUALIZATION ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

plot_metric_bars(
    suite_output,
    metrics=("rsrp_mean_dbm", "rsrp_p10_dbm", "sir_median_db", "sir_p10_db"),
    ax=axes[0],
)
axes[0].set_title("Metric Comparison")

plot_cdf(suite_output, metric="rsrp_values_dbm", ax=axes[1])
axes[1].set_title("RSRP CDF")

plt.suptitle("Motivating Example — Duke, 3.5 GHz", y=1.02)
plt.tight_layout()
plt.show()